In [ ]:
!wget -O PaDEL_Descriptors.csv https://huggingface.co/datasets/junhong1222/PaDEL_Descriptors/resolve/main/PaDEL_Descriptors.csv

In [1]:
! pip install scikit-learn==1.2.2 xgboost==1.7.3 numpy==1.26.4 pandas==2.2.3 PyTDC tqdm padelpy

  Using cached xgboost-1.7.3-py3-none-manylinux2014_x86_64.whl.metadata (1.9 kB)
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached pytdc-1.1.15-py3-none-any.whl
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached padelpy-0.1.14-py2.py3-none-any.whl.metadata (7.7 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached accelerate-0.33.0-py3-none-any.whl.metadata (18 kB)
  Using cached dataclasses-0.6-py3-none-any.whl.metadata (3.0 kB)
  Using cached datasets-2.19.2-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.2-py3-none-any.whl.metadata (9.3 kB)
  Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl.metadata (4.9 kB)

In [2]:
from tdc.benchmark_group import admet_group
from tqdm import tqdm
from xgboost import XGBRegressor
from calici_boost import *

In [3]:
group = admet_group(path = 'data/')
predictions_list = []

for seed in tqdm([1, 2, 3, 4, 5]):
    benchmark = group.get('Caco2_Wang') 
    # all benchmark names in a benchmark group are stored in group.dataset_names
    predictions = {}
    name = benchmark['name']
    train_val, test = benchmark['train_val'], benchmark['test']

    train_padel = add_padel_descriptors(train_val, 'train_val')
    test_padel = add_padel_descriptors(test, 'test')
    train_clean, test_clean = clean_data(train_padel, test_padel)

    x_train, y_train, x_test, y_test = featurize(train_clean, test_clean, seed)
    xgboost_model = XGBRegressor(
        **xg_parmas,
        max_bin=512,
        random_state=seed,
        tree_method='gpu_hist',
        gpu_id=0
    )
    xgboost_model.fit(x_train, y_train,
                        verbose=True)
    y_pred_test = xgboost_model.predict(x_test) 
    predictions[name] = y_pred_test
    predictions_list.append(predictions)

results = group.evaluate_many(predictions_list)
print(results)

Found local copy...
100%|██████████| 5/5 [00:13<00:00,  2.62s/it]

{'caco2_wang': [0.256, 0.006]}
